# Homework 5 [DRAFT] - Deployment


## Q1. Instalar `uv` y ver su versión

In [1]:
import subprocess, sys
r = subprocess.run([sys.executable, '-m', 'uv', '--version'], capture_output=True, text=True)
uv_version = r.stdout.strip()
print(uv_version)

uv 0.12.23 (46b84fd0b 2026-10-03 x86_64-pc-windows-msvc)


## Q2. Proyecto uv + scikit-learn 1.6.1



In [3]:
import re
text = open('uv_project/uv.lock', encoding='utf-8').read()
sec = re.search(r'name = "scikit-learn".*?(?=\n\[\[package\]\])', text, re.S).group(0)
hashes = re.findall(r'"(sha256:[0-9a-f]+)"', sec)
first_sdist = hashes[0]
first_wheel = hashes[1]
print('Primer hash (sdist):', first_sdist)
print('Primer hash (wheel):', first_wheel)

Primer hash (sdist): sha256:b4fc2525eca2c69a59260f583c56a7557c6ccdf8deafdba6e060f94c1c59738e
Primer hash (wheel): sha256:2ffa1e9e25b3d93990e74a4be2c2fc61ee5af85811562f1288d5d055880c4322



## Q3. Puntuar con el pipeline dado



In [5]:
import pickle, warnings, hashlib
warnings.filterwarnings('ignore')

md5 = hashlib.md5(open('pipeline_v1.bin', 'rb').read()).hexdigest()
print('md5:', md5)

with open('pipeline_v1.bin', 'rb') as f:
    pipeline = pickle.load(f)

record = {
    'lead_source': 'paid_ads',
    'number_of_courses_viewed': 2,
    'annual_income': 79276.0,
}
p = pipeline.predict_proba(record)[0][1]
print('Probabilidad:', round(p, 4))

md5: 7d17d2e4dfbaf1e408e1a62e6e880d49


Probabilidad: 0.5336



## Q4. Servir con FastAPI



In [7]:
import time, requests

proc = subprocess.Popen([sys.executable, '-m', 'uvicorn', 'serve:app', '--port', '8000'],
                        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(6)
try:
    client = {
        'lead_source': 'organic_search',
        'number_of_courses_viewed': 4,
        'annual_income': 80304.0,
    }
    resp = requests.post('http://localhost:8000/', json=client)
    res = resp.json()
    print(res)
finally:
    proc.terminate()

{'probability': 0.5340417283801275}



## Q5. Tamaño de la imagen base



In [9]:
pull = subprocess.run(['docker', 'pull', 'agrigorev/zoomcamp-model:2025'], capture_output=True, text=True)
print('pull ok:', pull.returncode == 0)

hist = subprocess.run(['docker', 'history', 'agrigorev/zoomcamp-model:2025', '--no-trunc', '--format', '{{.Size}}'],
                      capture_output=True, text=True).stdout
total = 0.0
for line in hist.splitlines():
    mm = re.match(r'(\d+(?:\.\d+)?)(kB|MB|GB)?', line.strip())
    if not mm:
        continue
    n = float(mm.group(1)); u = mm.group(2)
    total += (n / 1024) if u == 'kB' else (n * 1024) if u == 'GB' else n
print('Tamaño (capas sin comprimir):', round(total, 1), 'MB')

imgs = subprocess.run(['docker', 'images', 'agrigorev/zoomcamp-model:2025'], capture_output=True, text=True).stdout
print(imgs.strip())

pull ok: True
Tamaño (capas sin comprimir): 136.7 MB
IMAGE                           ID             DISK USAGE   CONTENT SIZE   EXTRA
agrigorev/zoomcamp-model:2025   14d79fde0bbf        181MB         44.3MB



## Q6. Contenedor Docker y puntuación



In [11]:
build = subprocess.run(['docker', 'build', '-t', 'hw5-lead', '.'], capture_output=True, text=True)
print('build ok:', build.returncode == 0)

subprocess.run(['docker', 'rm', '-f', 'hw5-run'], capture_output=True, text=True)
run = subprocess.Popen(['docker', 'run', '--name', 'hw5-run', '-p', '8000:8000', 'hw5-lead'],
                       stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(12)
try:
    client = {
        'lead_source': 'organic_search',
        'number_of_courses_viewed': 4,
        'annual_income': 80304.0,
    }
    r6 = requests.post('http://localhost:8000/', json=client)
    print(r6.json())
finally:
    subprocess.run(['docker', 'stop', 'hw5-run'], capture_output=True, text=True)
    subprocess.run(['docker', 'rm', 'hw5-run'], capture_output=True, text=True)

build ok: True


{'probability': 0.9933071490756734}
